In [ ]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.utils import shuffle
from sklearn.model_selection import cross_val_score, KFold
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler


In [ ]:
def select_optimal_components(X, Y, max_components=6, cv_splits=5, random_state=42):
    """用交叉验证选择最优PLS主成分数，返回最佳n_components及各成分对应的MSE"""
    mse_scores = []
    cv = KFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    for n_comp in range(1, max_components + 1):
        pls = PLSRegression(n_components=n_comp)
        scores = cross_val_score(pls, X, Y, cv=cv, scoring='neg_mean_squared_error')
        mse_scores.append(-np.mean(scores))
    best_n_comp = np.argmin(mse_scores) + 1
    return best_n_comp, mse_scores

In [ ]:
from matplotlib import rcParams

# 设置全局字体为 Times New Roman
rcParams['font.family'] = 'Times New Roman'

# --- 数据读取与准备 ---
fc_matrix = pd.read_csv(r"E:\aliyun_backup\muilt_disorders\11_pls\result\pls_weight\偏离连接_SCZ_top10.csv", header=None).values
row_idx, col_idx = np.triu_indices_from(fc_matrix, k=1)
nonzero_mask = fc_matrix[row_idx, col_idx] != 0

region_pairs = [f"{r}-{c}" for r, c in zip(row_idx[nonzero_mask], col_idx[nonzero_mask])]
fc_values = fc_matrix[row_idx[nonzero_mask], col_idx[nonzero_mask]]

# 根据权重绝对值排序并取前 500 个
top_500_indices = np.argsort(fc_values)[-500:]  # 取绝对值排序的后 500 个索引
top_500_region_pairs = [region_pairs[i] for i in top_500_indices]
top_500_fc_values = fc_values[top_500_indices]

# 构建 DataFrame
fc_df = pd.DataFrame({"Region_Pair": top_500_region_pairs, "FC_Value": top_500_fc_values})


# fc_df = pd.DataFrame({"Region_Pair": region_pairs, "FC_Value": fc_values})

nt_df = pd.read_csv(r"E:\aliyun_backup\muilt_disorders\15_Mitochondrial\Mitochondrial_contribution.csv")
merged_df = pd.merge(fc_df, nt_df, on="Region_Pair")

Y = merged_df["FC_Value"].values
X = merged_df.drop(columns=["Region_Pair", "FC_Value"]).values
genes = merged_df.drop(columns=["Region_Pair", "FC_Value"]).columns

# # 对 X 和 Y 进行归一化
# scaler_X = MinMaxScaler()
# scaler_Y = MinMaxScaler()

# X = scaler_X.fit_transform(X)
# Y = scaler_Y.fit_transform(Y.reshape(-1, 1)).flatten()

# --- 自动选择最佳主成分数量 ---
best_n_comp, mse_scores = select_optimal_components(X, Y, max_components=6, cv_splits=5)
print(f"最佳PLS主成分数量: {best_n_comp}")

# 可视化MSE曲线（可选）
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(mse_scores, marker='o')
plt.xlabel("Number of PLS components", fontsize=14)
plt.ylabel("CV Mean Squared Error", fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
# plt.title("PLS Components Selection by CV")
plt.savefig(r"E:\aliyun_backup\muilt_disorders\15_Mitochondrial\SCZ_pls_selection.tif", dpi=300)
plt.show()

# --- 用最佳成分数训练PLS ---
pls = PLSRegression(n_components=best_n_comp)
pls.fit(X, Y)
true_weights = pls.x_weights_[:, 0]
X_scores = pls.x_scores_
Y_scores = pls.y_scores_

explained_var_X = np.var(X_scores, axis=0) / np.sum(np.var(X, axis=0))
explained_var_Y = np.var(Y_scores, axis=0) / np.var(Y)
explained_df = pd.DataFrame({
    "Component": [f"PLS{i+1}" for i in range(best_n_comp)],
    "ExplainedVariance_X": explained_var_X,
    "ExplainedVariance_Y": explained_var_Y
})
print(explained_df)

# 绘制 explained_var_X 的折线图
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_var_X) + 1), explained_var_X, marker='o', linestyle='-', color='b', label='Explained Variance (X)')
plt.xlabel("PLS Component", fontsize=14)
plt.ylabel("Explained Variance", fontsize=14)
# plt.title("Explained Variance of X by PLS Components", fontsize=16)
plt.xticks(range(1, len(explained_var_X) + 1), fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.savefig(r"E:\aliyun_backup\muilt_disorders\15_Mitochondrial\SCZ_explained_variance_X.tif", dpi=300)
plt.show()

In [ ]:
# --- 置换检验P值计算（第一成分） ---
n_permutations = 1000
perm_weights = np.zeros((n_permutations, X.shape[1]))

# 使用最佳成分的解释方差
optimal_component_index = np.argmax(explained_var_X)  # 找到解释方差最大的成分索引
print(f"最佳成分索引: {optimal_component_index + 1}")

for i in tqdm(range(n_permutations), desc="Running permutations"):
    Y_perm = shuffle(Y, random_state=None)
    pls_perm = PLSRegression(n_components=best_n_comp)
    pls_perm.fit(X, Y_perm)
    perm_weights[i, :] = pls_perm.x_weights_[:, optimal_component_index]

p_values = []
for i in range(X.shape[1]):
    perm_dist = perm_weights[:, i]
    true_val = true_weights[i]
    p = np.mean(np.abs(perm_dist) >= np.abs(true_val))
    p_values.append(p)

result_df = pd.DataFrame({
    "Neurotransmitter": genes,
    "PLS1_Weight": true_weights,
    "P_value": p_values
}).sort_values(by="PLS1_Weight", key=np.abs, ascending=False)

print(result_df.head(10))
result_df.to_csv(r"E:\aliyun_backup\muilt_disorders\15_Mitochondrial\SCZ_PLS1_neurotransmitter_weights_with_pvals.csv", index=False)
# explained_df.to_csv("PLS_explained_variance.csv", index=False)